# Run the same source through Auto Loader and COPY INTO side by side and decide which fits each ingest pattern

The platform lead is two months into a "standardize the ingest stack" OKR and the team is split. The clickstream engineer wants Auto Loader because files land every minute; the data warehouse engineer wants COPY INTO because the daily batch CSV is the workload their team owns. You have one session to build both against the same UC Volume, capture the numbers (elapsed time, file-tracking behavior), and write the decision note the lead can paste into next Monday's eng review.

The hands-on work:

- Stand up one UC Volume with 12 baseline product JSON files.
- Ingest them into `bronze_autoloader` with Auto Loader and `trigger(availableNow=True)`. Re-run and confirm a second pass writes zero new rows.
- Ingest the same 12 files into `bronze_copyinto` with `COPY INTO`. Re-run and confirm a second pass writes zero new rows.
- Write a one-paragraph decision note into a `decision_note` table that names which pattern fits the clickstream (per-minute JSON) and which fits the daily batch CSV.

**Time.** Plan for about 65 minutes hands-on. Both ingest patterns finish in seconds on Free Edition; the slow part is the SQL warehouse cold start the first time COPY INTO runs. Budget up to 110 minutes for the session window.

**Cost.** Zero dollars. Both ingest paths fit inside the Free Edition daily compute quota. COPY INTO routes to the Starter SQL warehouse (counted against warehouse-seconds on the same quota); Auto Loader runs on serverless attached compute.

This lab is a **Compare-it** variant per LAB-CREATION-BLUEPRINT section 21: two competing ingest architectures against the same source, justified against two stated constraints (per-minute clickstream vs daily 3am batch).

This lab maps to Databricks DE Associate Domain 2: Data Ingestion and Loading (~20% of exam weight, provisional).

**Runtime note.** Auto Loader needs the in-notebook `spark` session, so this notebook must run inside a Databricks workspace notebook attached to serverless all-purpose compute. The setup cell guards against running outside Databricks.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. The comparison metric (elapsed seconds for
# each path) is captured into module-level globals during Tasks 2 and 3, then
# surfaced via Checkpoint 4's checkpointMeta in lib/labs.ts.

import atexit
import getpass
import io
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "auto-loader-vs-copy-into"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_ingest_compare"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

SOURCE_VOLUME = "source_files"
CHECKPOINTS_VOLUME = "checkpoints"
SOURCE_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{SOURCE_VOLUME}"
CHECKPOINTS_VOLUME_FQN = f"{LAB_SCHEMA_FQN}.{CHECKPOINTS_VOLUME}"

SOURCE_DATE_PREFIX = "2026-05-13"
SOURCE_VOLUME_PATH = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}/{SOURCE_DATE_PREFIX}"
CHECKPOINT_VOLUME_BASE = f"/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{CHECKPOINTS_VOLUME}"
AUTOLOADER_CHECKPOINT_PATH = f"{CHECKPOINT_VOLUME_BASE}/bronze_autoloader"

BRONZE_AUTOLOADER = "bronze_autoloader"
BRONZE_COPYINTO = "bronze_copyinto"
DECISION_NOTE_TABLE = "decision_note"
BRONZE_AUTOLOADER_FQN = f"{LAB_SCHEMA_FQN}.{BRONZE_AUTOLOADER}"
BRONZE_COPYINTO_FQN = f"{LAB_SCHEMA_FQN}.{BRONZE_COPYINTO}"
DECISION_NOTE_FQN = f"{LAB_SCHEMA_FQN}.{DECISION_NOTE_TABLE}"

NUM_FILES = 12

# Captured during Tasks 2 and 3 for the Option D comparisonMetric surface.
autoloader_elapsed_seconds = None
copyinto_elapsed_seconds = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, confirm Spark is available in-notebook.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")
print("Note: COPY INTO routes to this warehouse and may cold-start the first time.")
print("Asking the Starter warehouse to put on its reading glasses...")

try:
    spark  # type: ignore[name-defined]
except NameError:
    print("BLOCKED: this notebook must run inside a Databricks workspace.")
    print("Auto Loader needs the in-notebook spark session. Import this .ipynb")
    print("into Free Edition, attach to serverless all-purpose compute, and re-run.")
    raise SystemExit(1)
print("Spark session detected.")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Compare-it labs
# build two ingest stacks but share one cleanup pattern: tables first
# (decision_note last-created so it goes first), then volume contents, then
# volumes, then schema. The Starter SQL warehouse is shared and outside
# this lab's lifecycle, so the cleanup cell does NOT stop it.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_managed_table",
        id=DECISION_NOTE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {DECISION_NOTE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=BRONZE_COPYINTO_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BRONZE_COPYINTO_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=BRONZE_AUTOLOADER_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BRONZE_AUTOLOADER_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=CHECKPOINTS_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:{CHECKPOINT_VOLUME_BASE}/",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/{SOURCE_VOLUME}/",
    ),
    CleanupResource(
        type="uc_volume",
        id=CHECKPOINTS_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {CHECKPOINTS_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume",
        id=SOURCE_VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {SOURCE_VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                self._recursive_clear(volume_path)
            except Exception:
                pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def _recursive_clear(self, path):
        try:
            listing = list(w.files.list_directory_contents(path))
        except Exception:
            return
        for entry in listing:
            if entry.is_directory:
                self._recursive_clear(entry.path)
                try:
                    w.files.delete_directory(entry.path)
                except Exception:
                    pass
            else:
                try:
                    w.files.delete(entry.path)
                except Exception:
                    pass

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume":
            catalog, schema, volume = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.volumes "
                f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                f"AND volume_name = '{volume}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                return len(list(w.files.list_directory_contents(volume_path))) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
            (
                "SELECT catalog_name, schema_name, volume_name FROM system.information_schema.volume_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the schema, two volumes, the 12 source files, and both empty bronze tables

Two volumes (source files + Auto Loader checkpoints), 12 identical-schema JSON files, two empty bronze tables. The schema is the same as Lab 3 to keep the lesson focused on the comparison. The validator confirms both bronze tables exist with row count 0 before either ingest runs.

Build it in this order:

1. `CREATE SCHEMA workspace.default.labrun_ingest_compare` and tag it.
2. `CREATE VOLUME` source_files and checkpoints volumes; tag both.
3. Upload 12 JSON files under `/Volumes/.../source_files/2026-05-13/product-001.json` through `-012.json`. Each holds `{product_id, product_name, unit_price}`.
4. `CREATE TABLE` both bronze tables empty, with columns `product_id STRING, product_name STRING, unit_price DOUBLE`. Tag both. (Auto Loader can ingest into an empty pre-created table; COPY INTO can too.)

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, both volumes, 12 source files, two empty bronze tables.

# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Run ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

for vol_fqn in (SOURCE_VOLUME_FQN, CHECKPOINTS_VOLUME_FQN):
    # YOUR CODE: Run CREATE VOLUME IF NOT EXISTS vol_fqn (managed volume,
    # no LOCATION clause) via run_sql().
    # YOUR CODE: Run ALTER VOLUME vol_fqn SET TAGS
    # ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().
    pass

# Upload the 12 baseline files.
for idx in range(1, NUM_FILES + 1):
    file_name = f"product-{idx:03d}.json"
    file_path = f"{SOURCE_VOLUME_PATH}/{file_name}"
    product = {
        "product_id": f"P-{idx:03d}",
        "product_name": f"Product {idx}",
        "unit_price": round(50.0 + idx * 7.5, 2),
    }
    payload = json.dumps(product).encode("utf-8")
    # YOUR CODE: Upload the file via
    # w.files.upload(file_path=file_path, contents=io.BytesIO(payload),
    # overwrite=True).
    print(f"Uploaded {file_path}")

for table_fqn in (BRONZE_AUTOLOADER_FQN, BRONZE_COPYINTO_FQN):
    # YOUR CODE: Run CREATE TABLE IF NOT EXISTS table_fqn with three columns
    # (product_id STRING, product_name STRING, unit_price DOUBLE) USING DELTA
    # via run_sql().
    # YOUR CODE: Run ALTER TABLE table_fqn SET TAGS
    # ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().
    pass

print()
print(f"Source files staged under {SOURCE_VOLUME_PATH}/")
print(f"Both bronze tables created empty.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: 12 seed files staged, both bronze tables created empty.


def checkpoint_1(session):
    try:
        try:
            listing = list(w.files.list_directory_contents(SOURCE_VOLUME_PATH + "/"))
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not list {SOURCE_VOLUME_PATH}/: {exc}. "
                    f"Confirm the 12 baseline files uploaded."
                ),
            )
        json_files = [e for e in listing if e.path.endswith(".json")]
        if len(json_files) < NUM_FILES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Source path has {len(json_files)} *.json files; expected {NUM_FILES}. "
                    f"Re-run the upload loop."
                ),
            )

        for table_fqn, table_name in [
            (BRONZE_AUTOLOADER_FQN, BRONZE_AUTOLOADER),
            (BRONZE_COPYINTO_FQN, BRONZE_COPYINTO),
        ]:
            table_sql = (
                "SELECT table_type, data_source_format "
                "FROM system.information_schema.tables "
                f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
                f"AND table_name = '{table_name}'"
            )
            rows = run_sql(table_sql)["rows"]
            if not rows:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {table_fqn} not found. Did CREATE TABLE run?",
                )
            table_type, fmt = rows[0]
            if table_type != "MANAGED" or (fmt or "").upper() != "DELTA":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Table {table_fqn} is {table_type}/{fmt}; expected MANAGED/DELTA."
                    ),
                )
            count_rows = run_sql(f"SELECT count(*) FROM {table_fqn}")["rows"]
            row_count = int(count_rows[0][0]) if count_rows else 0
            if row_count != 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{table_fqn} has {row_count} rows; expected 0 at this checkpoint. "
                        f"The ingest tasks run after this checkpoint. Drop and recreate "
                        f"the table empty."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either says file count is wrong (upload loop did not run all 12), one of the bronze tables is missing (CREATE TABLE skipped), or a bronze table already has rows (you ran an ingest task before this checkpoint).

</details>

<details><summary>Hint 2 (direction)</summary>

Loop 12 times to upload, loop twice to create the two bronze tables. The bronze schema for both: `(product_id STRING, product_name STRING, unit_price DOUBLE) USING DELTA`. Match the column types to what the JSON parser will infer to avoid mismatches when COPY INTO runs.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

for vol_fqn in (SOURCE_VOLUME_FQN, CHECKPOINTS_VOLUME_FQN):
    run_sql(f"CREATE VOLUME IF NOT EXISTS {vol_fqn}")
    run_sql(f"ALTER VOLUME {vol_fqn} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

for idx in range(1, NUM_FILES + 1):
    file_path = f"{SOURCE_VOLUME_PATH}/product-{idx:03d}.json"
    product = {
        "product_id": f"P-{idx:03d}",
        "product_name": f"Product {idx}",
        "unit_price": round(50.0 + idx * 7.5, 2),
    }
    payload = json.dumps(product).encode("utf-8")
    w.files.upload(file_path=file_path, contents=io.BytesIO(payload), overwrite=True)

for table_fqn in (BRONZE_AUTOLOADER_FQN, BRONZE_COPYINTO_FQN):
    run_sql(
        f"CREATE TABLE IF NOT EXISTS {table_fqn} ("
        f"  product_id STRING, product_name STRING, unit_price DOUBLE"
        f") USING DELTA"
    )
    run_sql(f"ALTER TABLE {table_fqn} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
```

</details>

**Wallet check.** Still at $0.00. Twelve tiny JSON files plus two managed tables plus two volumes. Free Edition has not noticed.

## Task 2: Ingest via Auto Loader, then prove the second run is a no-op

Two passes:

1. First pass: Auto Loader stream with `trigger(availableNow=True)` against the source volume. After this run, `bronze_autoloader` should have 12 rows. Time the run so the elapsed seconds can be captured in `autoloader_elapsed_seconds`.

2. Second pass: re-run the EXACT same stream. The checkpoint location remembers which files Auto Loader has seen, so the second pass writes 0 new rows. `DESCRIBE HISTORY bronze_autoloader` should show two `WRITE` commits, the second with `numOutputRows=0`.

The Auto Loader contract: idempotency is tied to the `schemaLocation` / `checkpointLocation` state, NOT the bronze table name. If you delete the checkpoint state and re-run, Auto Loader treats every file as new.

In [ ]:
# NBVAL_SKIP
# Task 2: Auto Loader ingest, two passes. Capture elapsed time on the
# first pass for the comparisonMetric surface.

def run_autoloader():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", AUTOLOADER_CHECKPOINT_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(SOURCE_VOLUME_PATH + "/")
        .writeStream.option("checkpointLocation", AUTOLOADER_CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_AUTOLOADER_FQN)
    )

# YOUR CODE: First pass. Wrap run_autoloader() and .awaitTermination() in a
# time.perf_counter() block. Store the elapsed seconds in
# autoloader_elapsed_seconds (declared global so the value is visible to
# Checkpoint 4 via the module).

print(f"Auto Loader first pass: {autoloader_elapsed_seconds:.2f}s")

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_AUTOLOADER_FQN}")["rows"]
print(f"{BRONZE_AUTOLOADER_FQN} after first pass: {int(count_rows[0][0])} rows")

# YOUR CODE: Second pass. Same run_autoloader() call. The checkpoint
# location remembers what it has ingested, so this run should be a no-op.

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_AUTOLOADER_FQN}")["rows"]
print(f"{BRONZE_AUTOLOADER_FQN} after second pass: {int(count_rows[0][0])} rows (expect still 12)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: bronze_autoloader has 12 rows; DESCRIBE HISTORY shows at
# least two WRITE commits; the most recent WRITE wrote zero rows.


def checkpoint_2(session):
    try:
        count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_AUTOLOADER_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != NUM_FILES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_AUTOLOADER_FQN} has {row_count} rows; expected {NUM_FILES}. "
                    f"The first Auto Loader pass should have ingested all 12 files."
                ),
            )

        history = run_sql(
            f"SELECT version, operation, operationMetrics "
            f"FROM (DESCRIBE HISTORY {BRONZE_AUTOLOADER_FQN}) "
            f"ORDER BY version"
        )
        write_rows = [r for r in history["rows"] if str(r[1]) == "WRITE"]
        if len(write_rows) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DESCRIBE HISTORY shows only {len(write_rows)} WRITE commits; "
                    f"expected at least 2 (one per Auto Loader pass). Re-run the "
                    f"second pass."
                ),
            )
        last_write = write_rows[-1]
        metrics = last_write[2] or {}
        # operationMetrics is a map; the field name is 'numOutputRows'.
        num_rows_str = None
        if isinstance(metrics, dict):
            num_rows_str = metrics.get("numOutputRows")
        elif isinstance(metrics, str):
            try:
                num_rows_str = json.loads(metrics).get("numOutputRows")
            except Exception:
                num_rows_str = None
        try:
            num_rows = int(num_rows_str) if num_rows_str is not None else None
        except (TypeError, ValueError):
            num_rows = None
        if num_rows is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not parse numOutputRows from the latest WRITE commit's "
                    f"operationMetrics. Got: {metrics!r}."
                ),
            )
        if num_rows != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest WRITE commit wrote {num_rows} rows; expected 0 on the "
                    f"second Auto Loader pass. The checkpoint state may have been "
                    f"deleted between passes."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes. Either the row count is not 12 (the first pass did not ingest everything) or the second pass wrote new rows (the checkpoint state was cleared or pointed elsewhere).

</details>

<details><summary>Hint 2 (direction)</summary>

Two passes of the same stream. Wrap each in `time.perf_counter()` if you want to time them; assign the first pass's elapsed time to `autoloader_elapsed_seconds`. The `run_autoloader()` helper above gives you the stream; call `.awaitTermination()` on its return value. Run the cell twice (or call the helper twice in one cell).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
global autoloader_elapsed_seconds

start = time.perf_counter()
query = run_autoloader()
query.awaitTermination()
autoloader_elapsed_seconds = time.perf_counter() - start

# Second pass; the checkpoint state means it is a no-op.
second = run_autoloader()
second.awaitTermination()
```

</details>

**Wallet check.** Still at $0.00. Two trigger.availableNow passes against 12 tiny files. The elapsed-time numbers you captured are useful for the comparison; they do not move the wallet.

## Task 3: Ingest via COPY INTO, then prove the second run is a no-op

COPY INTO is a SQL statement, not a streaming query. It runs on the Starter SQL warehouse. The statement:

```sql
COPY INTO workspace.default.labrun_ingest_compare.bronze_copyinto
FROM '/Volumes/.../source_files/2026-05-13/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

Two passes:

1. First pass: 12 rows land in `bronze_copyinto`. Time it; the warehouse cold-start can dominate on the first call.
2. Second pass: same statement, same FROM. COPY INTO tracks ingested filenames in its internal metadata; the second pass returns `num_inserted_rows=0`.

If the warehouse is cold, the first pass can take 30 to 90 seconds. Re-running the cell wakes it up.

In [ ]:
# NBVAL_SKIP
# Task 3: COPY INTO ingest, two passes. Time the first pass for the
# comparisonMetric surface.

copy_into_sql = (
    f"COPY INTO {BRONZE_COPYINTO_FQN} "
    f"FROM '{SOURCE_VOLUME_PATH}/' "
    f"FILEFORMAT = JSON "
    f"FORMAT_OPTIONS ('inferSchema' = 'true') "
    f"COPY_OPTIONS ('mergeSchema' = 'true')"
)

# YOUR CODE: First pass. Wrap run_sql(copy_into_sql) in time.perf_counter()
# and store the elapsed seconds in copyinto_elapsed_seconds.

print(f"COPY INTO first pass: {copyinto_elapsed_seconds:.2f}s")

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_COPYINTO_FQN}")["rows"]
print(f"{BRONZE_COPYINTO_FQN} after first pass: {int(count_rows[0][0])} rows")

# YOUR CODE: Second pass. Same run_sql(copy_into_sql) call. COPY INTO's
# internal file-tracking ledger makes this a no-op.

count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_COPYINTO_FQN}")["rows"]
print(f"{BRONZE_COPYINTO_FQN} after second pass: {int(count_rows[0][0])} rows (expect still 12)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: bronze_copyinto has 12 rows; DESCRIBE HISTORY shows two
# WRITE commits (or one WRITE and one COPY INTO depending on engine
# version), the latest of which inserted 0 rows.


def checkpoint_3(session):
    try:
        count_rows = run_sql(f"SELECT count(*) FROM {BRONZE_COPYINTO_FQN}")["rows"]
        row_count = int(count_rows[0][0]) if count_rows else 0
        if row_count != NUM_FILES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{BRONZE_COPYINTO_FQN} has {row_count} rows; expected {NUM_FILES}. "
                    f"Confirm the first COPY INTO inserted all 12 files."
                ),
            )

        history = run_sql(
            f"SELECT version, operation, operationMetrics "
            f"FROM (DESCRIBE HISTORY {BRONZE_COPYINTO_FQN}) "
            f"ORDER BY version"
        )
        # Accept any of WRITE, COPY INTO, or COPY INTO TABLE as the
        # data-mutating operation; Databricks engine versions report this
        # differently.
        data_ops = [
            r for r in history["rows"]
            if str(r[1]).upper().startswith("WRITE")
            or str(r[1]).upper().startswith("COPY")
        ]
        if len(data_ops) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DESCRIBE HISTORY shows only {len(data_ops)} ingest commits; "
                    f"expected at least 2 (one per COPY INTO pass). Re-run the "
                    f"second pass."
                ),
            )

        last_op = data_ops[-1]
        metrics = last_op[2] or {}
        candidate_keys = ("numOutputRows", "num_inserted_rows", "numTargetRowsInserted")
        num_rows = None
        if isinstance(metrics, dict):
            for key in candidate_keys:
                if key in metrics:
                    try:
                        num_rows = int(metrics[key])
                    except (TypeError, ValueError):
                        num_rows = None
                    break
        elif isinstance(metrics, str):
            try:
                parsed = json.loads(metrics)
                for key in candidate_keys:
                    if key in parsed:
                        try:
                            num_rows = int(parsed[key])
                        except (TypeError, ValueError):
                            num_rows = None
                        break
            except Exception:
                num_rows = None

        if num_rows is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not parse insert-row metrics from the latest commit. "
                    f"operationMetrics: {metrics!r}."
                ),
            )
        if num_rows != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest COPY INTO commit inserted {num_rows} rows; expected 0 "
                    f"on the second pass. The internal file-tracking ledger may have "
                    f"been cleared, or the FROM path changed between passes."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two failure shapes. Either the row count is not 12 (the first COPY INTO did not land everything; check the FROM path is correct) or the second pass inserted new rows (the FROM path was different between passes, or you used `force = true`).

</details>

<details><summary>Hint 2 (direction)</summary>

One SQL statement, run twice. Wrap each call in `time.perf_counter()` to capture elapsed seconds. Use the `copy_into_sql` constant from this cell; do NOT change it between passes. The FROM clause is the directory path, with the trailing slash.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
global copyinto_elapsed_seconds

start = time.perf_counter()
run_sql(copy_into_sql)
copyinto_elapsed_seconds = time.perf_counter() - start

# Second pass; COPY INTO's internal file-tracking ledger makes this a no-op.
run_sql(copy_into_sql)
```

</details>

**Wallet check.** Still at $0.00. COPY INTO ran on the Starter warehouse for a handful of seconds total (warm pass) plus whatever the first cold-start added. Free Edition counts warehouse-seconds against the same daily quota; the wallet still says zero dollars.

## Task 4: Write the decision note

The platform lead needs one paragraph they can paste into next Monday's eng review. The decision_note table holds one row with a `note` column; the validator checks that the note text contains four anchor concepts (lowercased):

- `auto loader`
- `copy into`
- one of: `clickstream` or `minute`
- one of: `daily`, `batch`, or `3am`

The validator does NOT grade prose quality. It checks that the four anchors are present. Write the paragraph as honestly as you can; the reflection MCQ tests the comparative reasoning that the prose hints at.

Build it in this order:

1. Create the `decision_note` table with one column `note STRING`. Tag it.
2. `INSERT INTO ... VALUES ('<your paragraph here>')`. SQL string literals use single quotes; double any single quotes inside your prose.

In [ ]:
# NBVAL_SKIP
# Task 4: Create the decision_note table and insert your paragraph.

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS DECISION_NOTE_FQN with one
# column (note STRING) USING DELTA via run_sql().

# YOUR CODE: Run ALTER TABLE DECISION_NOTE_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

decision_note_text = (
    "Use Auto Loader for the clickstream because files land every minute and "
    "schemaEvolutionMode=addNewColumns handles unannounced fields. Use COPY INTO "
    "for the daily batch CSV at 3am because the schema is stable, a single "
    "scheduled statement is the simplest control plane, and the file-tracking "
    "ledger makes a missed-night replay safe."
)

# YOUR CODE: Run INSERT INTO DECISION_NOTE_FQN VALUES ('<decision_note_text>')
# via run_sql(). Escape any single quotes in the text by doubling them
# (e.g., "team's" becomes "team''s").

print(f"Decision note recorded in {DECISION_NOTE_FQN}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: decision_note has exactly one row; the note text contains
# the four anchor concepts (auto loader, copy into, a clickstream/minute
# term, and a daily/batch/3am term). Captures the elapsed-time comparison
# metric in the error_reason on PASS (the comparisonMetric surfaces via
# lib/labs.ts on the Option D card).


def checkpoint_4(session):
    try:
        rows = run_sql(f"SELECT note FROM {DECISION_NOTE_FQN}")["rows"]
        if len(rows) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{DECISION_NOTE_FQN} has {len(rows)} rows; expected exactly 1. "
                    f"Truncate and re-insert if you have more than one row."
                ),
            )
        note = str(rows[0][0] or "").lower()
        missing = []
        if "auto loader" not in note:
            missing.append("'auto loader'")
        if "copy into" not in note:
            missing.append("'copy into'")
        if not any(token in note for token in ("clickstream", "minute")):
            missing.append("one of 'clickstream' or 'minute' (the streaming source)")
        if not any(token in note for token in ("daily", "batch", "3am")):
            missing.append("one of 'daily', 'batch', or '3am' (the batch source)")
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Decision note is missing required anchor terms: "
                    + ", ".join(missing)
                    + ". Re-write the note to name both ingest patterns and both "
                    "source cadences."
                ),
            )

        if autoloader_elapsed_seconds is not None and copyinto_elapsed_seconds is not None:
            print(
                f"comparisonMetric capture: Auto Loader ingest of {NUM_FILES} files: "
                f"{autoloader_elapsed_seconds:.2f}s; COPY INTO ingest of the same "
                f"{NUM_FILES} files via Starter warehouse: {copyinto_elapsed_seconds:.2f}s. "
                f"Re-run is a no-op for both."
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The validator says which anchor is missing. The four concepts are: name Auto Loader, name COPY INTO, name the streaming source (clickstream or minute), name the batch source (daily, batch, or 3am).

</details>

<details><summary>Hint 2 (direction)</summary>

Three SQL statements: CREATE TABLE, ALTER TABLE SET TAGS, INSERT INTO. The note text can be inlined into the INSERT, or built as a Python f-string and substituted. Double any single quotes inside the prose before passing to SQL.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
run_sql(
    f"CREATE TABLE IF NOT EXISTS {DECISION_NOTE_FQN} (note STRING) USING DELTA"
)
run_sql(f"ALTER TABLE {DECISION_NOTE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

escaped = decision_note_text.replace("'", "''")
run_sql(f"INSERT INTO {DECISION_NOTE_FQN} VALUES ('{escaped}')")
```

</details>

**Wallet check.** Still at $0.00. One small Delta table with a single row of prose. The session has spent a sliver of the daily compute quota and zero dollars.

## Cleanup

Time to tear it all down. The cell below drops both bronze tables and the decision_note table, purges both volumes, drops the volumes, then drops the schema with `CASCADE`. The Starter SQL warehouse is shared and stays running for other workloads. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_destroyed = 0
standard_destroyed = len(CLEANUP_MANIFEST) - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Two ingest stacks against twelve tiny JSON files, plus one paragraph of decision prose. Free Edition swallowed every byte and a handful of warehouse-seconds. The expensive thing this session would have prevented is the platform lead's six-month "standardize the ingest stack" project drifting into Q4 because the team could not align on Auto Loader vs COPY INTO.

## Reflection

These are not graded. They are for you.

1. Walk through the file-tracking ledger each ingest pattern maintains. Auto Loader uses the schemaLocation and a checkpoint location to remember which files it has processed; COPY INTO uses an internal metadata table keyed by filename plus modification time. Sketch a scenario where Auto Loader and COPY INTO would disagree on whether a file is "new."

2. Your team is moving 200 GB of historical CSV files into a Unity Catalog table as a one-time backfill, then standing up a steady-state stream for the daily delta. Which pattern fits each phase (backfill, steady-state)? Justify each pick in terms of idempotency, schema evolution exposure, and operational overhead.

## Exam-Style Practice

**Question 1 (Domain 2, Auto Loader vs COPY INTO selection):**

A retail platform ingests product files from an external partner. The files land every minute, contain JSON with occasional new optional fields, and the team wants schema evolution to happen automatically without manual ALTER TABLE intervention. Which ingest mechanism best fits?

A. `COPY INTO` with `FORMAT_OPTIONS ('inferSchema' = 'true', 'mergeSchema' = 'true')`.

B. Auto Loader with `cloudFiles.schemaLocation` set and `cloudFiles.schemaEvolutionMode=addNewColumns`.

C. `CREATE TABLE ... USING DELTA LOCATION '<volume_path>'` followed by `MSCK REPAIR TABLE` on a cron.

D. A nightly `INSERT INTO` from a `read_files('<path>', format => 'json')` view.

<details><summary>Show answer</summary>

**Correct: B.**

- A would work for one-shot loads with schema merge but COPY INTO does not auto-evolve a target table's schema on every run; mergeSchema applies per statement and reverts. Used for high-frequency streaming ingest with evolution, it requires extra orchestration.
- B is correct: Auto Loader with `addNewColumns` is the documented Databricks pattern for files-land-frequently-with-new-columns. The schemaLocation persists across runs, the addNewColumns mode auto-restarts the stream when a new column lands, and the bronze table evolves without intervention.
- C is wrong: `MSCK REPAIR TABLE` is a Hive partition-discovery command not used for Delta; Delta does not need it.
- D works for batch but does not provide idempotency on file-by-file basis; it re-reads the full directory every night and an INSERT INTO would duplicate every row unless the engineer added a watermark column.

</details>

**Question 2 (Domain 2, Compare-it; constraint-driven selection):**

A data team must ingest two sources into a Unity-Catalog-governed bronze layer:

- Source X: 100 small CSVs land in cloud storage every 24 hours at 3am UTC, schema is stable, monthly schema changes are coordinated.
- Source Y: a clickstream drops 5 to 20 small JSON files per minute, the upstream SDK occasionally adds optional fields without notice.

The team wants the simplest production-grade setup that meets both sources' needs with the lowest operational burden. Which combination is best?

A. Auto Loader for both, because Auto Loader handles both batch and streaming.

B. COPY INTO scheduled at 3am for Source X, Auto Loader with `trigger(availableNow=True)` (or `continuous` if upgraded) plus `schemaEvolutionMode=addNewColumns` for Source Y.

C. COPY INTO for both, because COPY INTO is idempotent and the simpler command surface.

D. A custom Spark Structured Streaming job for Source X and a Lakeflow Connect managed connector for Source Y.

<details><summary>Show answer</summary>

**Correct: B.**

- A would work but is unnecessarily heavy for Source X. The batch CSV at 3am does not need Auto Loader's continuous schemaLocation overhead; a daily COPY INTO is simpler and the schema-change coordination is already in the team's process.
- B is correct: COPY INTO fits the predictable daily batch with stable schema (single statement, scheduled via Lakeflow Jobs); Auto Loader fits the high-frequency stream with unpredictable column additions (schemaLocation + addNewColumns). This is the documented per-source decision.
- C is wrong: COPY INTO without scheduling magic does not handle the per-minute clickstream cadence well, and the schema evolution story for unpredictable columns is weaker than Auto Loader's.
- D is over-engineered; the team gains nothing from a custom Structured Streaming job over Auto Loader, and Lakeflow Connect managed connectors apply when the source is an external system (SaaS app, database), not a folder of files in cloud storage.

</details>